In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Reinforcement learning on a language model

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part C (application), session 1 of 2**
    - Part A built the value-based path to DQN; Part B built the policy-gradient methods up to PPO and GRPO
    - **the destination we start from:** the methods so far were shown on small environments; PPO was adopted for language models (in RLHF, reinforcement learning from human feedback) and GRPO was designed for them
    - the next session (C2) introduces preference data, one source of the reward used here
- **The question:** how does a language model become a policy, where does its reward come from, and which choices from Part B's slot table does it make?
- **By the end you can**
    - say what the state, action, episode and reward are for a language model
    - name who can produce a reward: a program, a learned reward model, a language-model judge
    - read one training step of RL on a language model as the Part B loop, and say which slot-table choices it makes
    - say why supervised fine-tuning usually comes before RL, and what the KL penalty is for


# A language model is a policy

**Why this section?** Everything in Part B needs a policy $\pi_\theta(\act \mid \state)$, states, actions and episodes. A language model already has all four, once we name them.

- the language-modeling task: given an input $\x$ (the prompt), produce an output $\y$, one token at a time, each conditioned on the tokens before it
- as reinforcement learning
    - **state:** the prompt and the response so far, $\stateseq_\tt = (\x, \y_{<\tt})$, the prompt and the tokens before position $\tt$ (one state per prefix; the first state is the prompt alone)
    - **action:** choose the next token from the vocabulary
    - **policy:** $\pi_\theta(\y_\tt \mid \x, \y_{<\tt})$: the model's next-token distribution
    - **episode:** the whole response, with $|\y|$ steps (the episode length $|\tau|$ of earlier sessions)
- the state is huge and the action set is the whole vocabulary: a table is impossible; the policy must be a network (Part A's limitation, Part B's answer)


**A response as an episode**

<img src="images/llm_episode.png" width=95%>

- the *state* is the prompt and the tokens generated so far; the *action* is the next token
- the whole response is one episode; the reward usually arrives only at the end (sparse), as on Frozen Lake
- in single-turn generation the transition is trivial: the next state is the old state plus the chosen token (no randomness from an environment); the episode ends when the model emits an end-of-response token
- the first state is the prompt alone, before any token


## Where does the reward come from?

**Why this section?** A policy gradient needs a reward for each response. Classify by who produces the label.

| Label produced by | Example | Strength | Weakness |
|:--|:--|:--|:--|
| a **program** (a **verifier**) | the final number of a math problem; unit tests for code | exact on what it checks, cheap, no human labels per response (but someone writes the checker, and it can accept wrong answers) | only for checkable tasks |
| a **learned model** (a **reward model**) | trained on human comparisons of chat answers | covers open-ended tasks | an imperfect proxy |
| a **language model as judge** | scores or ranks responses against a rubric | flexible, no training | has its own biases (for example, favoring long answers) |

- the reward is only a proxy for what we want, so the policy can learn to exploit it: **reward hacking** is getting a high reward without the quality the reward was meant to measure (the tour's Microduck found an *unexpected strategy*; that counts as hacking only if the reward is earned in a way the designers did not want)
    - for example, a model trained against a reward model may learn long, confident answers that score well but are not better
- a reward model is itself learned from **preference data**: the next session


## One training step on a language model

**Why this section?** This is the PPO or GRPO loop of Part B, with the language-model names filled in.

- **Sample:** for a prompt $\x$, draw a group of $G$ responses $\y^{(1)}, \ldots, \y^{(G)}$ from the current policy $\pi_\theta$
- **Score:** each response gets one reward $\rew(\x, \y)$, from a verifier, a reward model or a judge
- **Credit:** turn the rewards into advantages
    - GRPO: each reward minus the group mean, divided by the group standard deviation (sample standard deviation, divisor $G - 1$)
    - PPO: a learned critic and GAE (generalized advantage estimation, Part B session 2 appendix)
    - GRPO gives one number per response, shared by all of its tokens; PPO's advantage varies by token
    - by numbers (GRPO): $G = 4$ rewards $(1, 0, 0, 0)$ give mean $0.25$, advantages $1.5$ and $-0.5$ (three times); rewards $(1, 1, 1, 1)$ give advantages $0$
- **Update:** a clipped policy-gradient step that raises the probability of the tokens of high-advantage responses and lowers it for low-advantage ones
    - usually plus a **KL penalty** to a reference model (the next slide but one; some recipes set its weight to zero)
- repeat with fresh samples


## Which choices from the slot table?

**Why this section?** Part B ended with one theorem and three slots. A language model fills each of them.

| Slot | Choice for a language model | Why |
|:--|:--|:--|
| 1: weight | group-relative advantage (GRPO), or GAE from a critic (PPO) | one reward per response, sparse and at the end |
| 2: how far | clipped ratio, several epochs per batch | each batch of generations is expensive to produce (Part B, session 3) |
| 3: baseline source | the group mean (GRPO), or a critic (PPO) | a critic over every prefix is a second large network (Part B, session 4) |

- (these four are *threads* from Part B, not slots)
- on-policy versus off-policy: batches are sampled from the current policy, then reused a few epochs with the ratio (slightly off-policy after the first epoch)
- bias and variance: a verifier is exact on what it checks (but may accept wrong answers); the group baseline shrinks the advantage by the factor $(G-1)/G$ (a rescaling, closer to 1 as $G$ grows) and is noisier than a critic, which has its own bias
- evaluate versus improve: PPO keeps a critic as the evaluator; GRPO drops it
- exploration: sampling several responses from the policy; a group needs responses that differ
- **why GRPO fits:** the reward is one verifiable number per response, which needs no per-token credit (the price: coarse, whole-response credit), and there is no critic to train


## The KL penalty and the reference model

**Why this section?** The reward is a proxy (reward hacking), and the policy starts from a model we already like. We want to improve without wandering.

- the **reference model** $\pi_{\text{ref}}$ is a frozen copy of the model before RL (for example the supervised fine-tuned model)
- the **KL penalty** subtracts $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$ from the objective
    - $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$ is the average, over text sampled from the current policy, of $\log \pi_\theta - \log \pi_{\text{ref}}$: zero when the two agree, larger as they differ
    - large $\beta$: stay close to the reference; small $\beta$: chase the reward
    - it limits how far the policy can drift toward a hack, and tends to keep the text near the reference's fluency (it does not prevent hacks that stay close to the reference)
- how it differs from clipping (Part B, session 3): clipping limits the move from the old policy to the new one in a single iteration; the KL penalty anchors to one *fixed* model across the whole run
- in the loop, it is estimated on the sampled tokens (Part B, session 4's pseudo-code); RLHF-style PPO often folds a per-token KL into the *reward* instead
- it is not part of the original PPO paper (which has a different KL, to the previous policy); it comes from RLHF practice


## Why supervised fine-tuning usually comes before RL

**Why this section?** A condition for starting RL: the reward must be reachable.

- RL **raises the probability of responses the model already produces** some of the time and that earn reward
- if no sampled response earns reward (a binary verifier reward), there is no signal
    - in GRPO, a group with identical rewards (all failures, or all successes) has every advantage equal to $0$ (with a small constant in the denominator)
- so a model that rarely produces the required format is unlikely to find it by RL
    - **supervised fine-tuning** (SFT: train on example responses) first puts that behavior within reach
- this is a condition, not a law
    - a strong base model may already succeed often enough to start RL directly (some recent reasoning models did)
- SFT also supplies the fluent reference model for the KL penalty
- a check before spending compute: sample $G$ responses for a few prompts; many groups should contain both a success and a failure (a *hit* is a group with at least one success)
    - with success chance $p$ per response, a group contains a hit with probability $1 - (1-p)^G$, about $G p$ when that is small (for $p = 0.01$, $G = 8$: about $8\%$)


## What this costs, and what can go wrong

**Why this section?** The limitations that motivate the rest of the course.

- **generation is the cost:** a group of long responses per prompt, every iteration
- **the reward is a proxy:** hacking, judge biases, verifiers that accept wrong answers
- **sparse credit:** one reward for a long response, and (in GRPO) one weight for all of its tokens
- **no signal on easy or impossible prompts:** all-success and all-failure groups
- **preference data** is the route when we cannot write a verifier: the next session


## Aside: the probability of a whole response

- the policy factorizes over tokens: $\pi_\theta(\y \mid \x) = \prod_{\tt=1}^{|\y|} \pi_\theta(\y_\tt \mid \x, \y_{<\tt})$
- this is the probability of the *entire* episode, and is the one number that preference methods such as DPO (next session) compare between a preferred and a non-preferred response
- the policy-gradient methods of Part B work token by token and do not need it


# What we established, and what it sets up

- **Established**
    - a language model is a policy: state = prompt plus response so far, action = next token, a response is an episode
    - the reward is usually sparse (at the end) and can come from a program, a learned reward model or a judge: each a proxy that can be hacked
    - one RL step on a language model is the Part B loop: sample a group, score, turn rewards into advantages, take a clipped step with a KL penalty to a reference model
    - RL amplifies what the model already does, so supervised fine-tuning usually comes first
- **Sets up**
    - next session: preference data, how comparisons become a reward model, and where direct preference optimization (DPO) stands relative to policy gradients
    - the Advanced course on reinforcement fine-tuning applies all of this in detail
